In [ ]:
#################################################################
# 모델 7: MLP Classifier (신경망) (RandomizedSearchCV로 변경)
#################################################################
import numpy as np
from time import time
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning

# 모듈 임포트
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV # GridSearchCV -> RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier # Scikit-Learn의 신경망
from scipy.stats import uniform, randint # RandomizedSearchCV용

# 경고 메시지 무시
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

print("--- 0. 공통 모듈 임포트 완료 ---")

--- 0. 공통 모듈 임포트 완료 ---


In [ ]:
# --- 1. Original MNIST 데이터셋 로딩 (70,000개) ---
print("\n--- 1. Original MNIST 데이터셋 로딩 (70,000개) ---")
start_time = time()
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X = mnist.data/255
y = mnist.target.astype(np.uint8)
print(f"   ...로딩 완료. (소요 시간: {time() - start_time:.2f}초)")

print("\n--- 모델 7: MLP Classifier (신경망) ---")


--- 1. Original MNIST 데이터셋 로딩 (70,000개) ---
   ...로딩 완료. (소요 시간: 2.15초)

--- 모델 7: MLP Classifier (신경망) ---


   ...로딩 완료. (소요 시간: 2.15초)

--- 모델 7: MLP Classifier (신경망) ---


In [ ]:
# --- 7-1. 스케일링 테스트 ---
# 신경망(MLP)은 스케일링이 필수적이므로 파이프라인에 StandardScaler를 고정합니다.
print("   7-1. 스케일링 테스트: 불필요 (파이프라인에 StandardScaler 고정)")

   7-1. 스케일링 테스트: 불필요 (파이프라인에 StandardScaler 고정)


In [ ]:
# --- 7-2. 하이퍼파라미터 튜닝 (RandomizedSearchCV, 전체 X, y 사용) ---
print("   7-2. RandomizedSearchCV 튜닝 시작 (전체 데이터 사용, 시간이 다소 걸릴 수 있음)...")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MLPClassifier(random_state=42, max_iter=500)) # 수렴을 위해 max_iter 증가
])

# (신규) 더 다양해진 파라미터 "분포"
param_distribs = {
    # 요청하신 대로 은닉층 구조를 더 다양하게 탐색합니다.
    'model__hidden_layer_sizes': [
        (50,), (100,), (150,),
        (50, 50), (100, 50), (100, 100)
    ],
    'model__activation': ['tanh', 'relu'], # 활성화 함수
    'model__alpha': uniform(0.0001, 0.01), # L2 규제 (랜덤 실수)
    'model__learning_rate_init': uniform(0.001, 0.01) # 학습률 (랜덤 실수)
}

# n_iter=10: 위 분포에서 10개의 조합만 랜덤하게 테스트
rnd_search_mlp = RandomizedSearchCV(pipeline, param_distribs, n_iter=10, cv=3,
                                  scoring='accuracy', n_jobs=-1, verbose=2, random_state=42)
start_time = time()
rnd_search_mlp.fit(X, y)
print(f"   ...튜닝 완료. (소요 시간: {time() - start_time:.2f}초)")
print(f"   - 최적 파라미터: {rnd_search_mlp.best_params_}")
print(f"   - 최고 정확도: {rnd_search_mlp.best_score_:.4f}")

   7-2. RandomizedSearchCV 튜닝 시작 (전체 데이터 사용, 시간이 다소 걸릴 수 있음)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END model__activation=relu, model__alpha=0.0022233911067827614, model__hidden_layer_sizes=(50, 50), model__learning_rate_init=0.002834045098534338; total time= 1.1min
[CV] END model__activation=relu, model__alpha=0.00621653160488281, model__hidden_layer_sizes=(100, 50), model__learning_rate_init=0.005319450186421158; total time= 2.2min


Exception ignored in: <function ResourceTracker.__del__ at 0x70d1d4296980>
Traceback (most recent call last):
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END model__activation=tanh, model__alpha=0.0003058449429580245, model__hidden_layer_sizes=(100,), model__learning_rate_init=0.008219987722668248; total time= 3.8min


Exception ignored in: <function ResourceTracker.__del__ at 0x7d44d4096980>
Traceback (most recent call last):
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END model__activation=tanh, model__alpha=0.0016601864044243652, model__hidden_layer_sizes=(150,), model__learning_rate_init=0.001999749158180029; total time= 4.2min


...

Exception ignored in: <function ResourceTracker.__del__ at 0x7161b349a980>
Traceback (most recent call last):
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV] END model__activation=tanh, model__alpha=0.008065429868602328, model__hidden_layer_sizes=(150,), model__learning_rate_init=0.008796910002727693; total time= 1.3min
[CV] END model__activation=tanh, model__alpha=0.005347746602583892, model__hidden_layer_sizes=(100,), model__learning_rate_init=0.0014666566321361544; total time= 4.1min


Exception ignored in: <function ResourceTracker.__del__ at 0x712b1658a980>
Traceback (most recent call last):
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/yc54616/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


   ...튜닝 완료. (소요 시간: 1748.74초)
   - 최적 파라미터: {'model__activation': 'relu', 'model__alpha': np.float64(0.003763618432936917), 'model__hidden_layer_sizes': (100, 100), 'model__learning_rate_init': np.float64(0.0019060643453282082)}
   - 최고 정확도: 0.9688


   ...튜닝 완료. (소요 시간: 222.25초)
   - 최적 파라미터: {'model__activation': 'relu', 'model__alpha': np.float64(0.003763618432936917), 'model__hidden_layer_sizes': (100, 100), 'model__learning_rate_init': np.float64(0.0019060643453282082)}
   - 최고 정확도: 0.9688


In [ ]:
# --- 7-3. 최적 모델 저장 ---
model_filename = '07_mlp_classifier_best_0-1.joblib'
joblib.dump(rnd_search_mlp.best_estimator_, model_filename)
print(f"   ...최적 모델을 '{model_filename}' 파일로 저장했습니다.")

   ...최적 모델을 '07_mlp_classifier_best_0-1.joblib' 파일로 저장했습니다.
